# What adata contains what

The files are listed in the order of creation in "CustomBinning.ipynb".

In [1]:
import scanpy as sc
import os
import geopandas as gpd
import pandas as pd

data_path = "Cell_Segmentation/output/CRC_P2/002um/custom_binning"
data_path2 = "Cell_Segmentation/output/merged_custom_binning/002um"

**1.**  
**filtered_custom_bins.h5ad**  
We filter out any barcodes (bins), that are either in no nucleus or are in multiple nuclei at once (overlapping).
The resulting matrix contains all expressed genes per **bin**, not per nucleus.
- `.X`: rows (obs): Barcodes per one nucleus, Columns (var):	Genes from the 10x matrix, Values: Integer UMI counts
- `.obs`: 'in_tissue', 'array_row', 'array_col', 'pxl_row_in_fullres', 'pxl_col_in_fullres', index_x', 'index_y', 'id', 'is_within_polygon', 'is_not_in_an_polygon_overlap', 'Slide_ID'
- `.var`: 'gene_ids', 'feature_types', 'genome'

In [18]:
filtered_custom_bins = sc.read_h5ad(os.path.join(data_path, "filtered_custom_bins.h5ad"))
filtered_custom_bins

/home/charlottewolf/miniconda3/envs/NucleiSeg/lib/python3.11/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 1721528 × 18085
    obs: 'in_tissue', 'array_row', 'array_col', 'pxl_row_in_fullres', 'pxl_col_in_fullres', 'index_x', 'index_y', 'id', 'is_within_polygon', 'is_not_in_an_polygon_overlap', 'Slide_ID'
    var: 'gene_ids', 'feature_types', 'genome'

In [19]:
filtered_custom_bins.obs

,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres,index_x,index_y,id,is_within_polygon,is_not_in_an_polygon_overlap,Slide_ID
s_002um_02043_00141-1,1,2043,141,49055.614563,45360.824978,s_002um_02043_00141-1,s_002um_02043_00141-1,ID_1318569,True,True,CRC_P5
s_002um_03246_01880-1,1,3246,1880,40320.322188,58100.258125,s_002um_03246_01880-1,s_002um_03246_01880-1,ID_1056638,True,True,CRC_P5
s_002um_01050_01530-1,1,1050,1530,56350.990878,55477.439409,s_002um_01050_01530-1,s_002um_01050_01530-1,ID_1620019,True,True,CRC_P5
s_002um_01057_00712-1,1,1057,712,56275.268082,49502.109602,s_002um_01057_00712-1,s_002um_01057_00712-1,ID_1598983,True,True,CRC_P5
s_002um_00737_02449-1,1,737,2449,58665.104485,62181.512805,s_002um_00737_02449-1,s_002um_00737_02449-1,ID_1735286,True,True,CRC_P5
...,...,...,...,...,...,...,...,...,...,...,...
s_002um_01930_00297-1,1,1930,297,49885.719742,46496.957741,s_002um_01930_00297-1,s_002um_01930_00297-1,ID_1312732,True,True,CRC_P5
s_002um_00635_02644-1,1,635,2644,59416.094674,63602.988400,s_002um_00635_02644-1,s_002um_00635_02644-1,ID_1736877,True,True,CRC_P5
s_002um_00080_00956-1,1,80,956,63419.638408,51255.052050,s_002um_00080_00956-1,s_002um_00080_00956-1,ID_1755061,True,True,CRC_P5
s_002um_02017_00654-1,1,2017,654,49260.921104,49107.423831,s_002um_02017_00654-1,s_002um_02017_00654-1,ID_1321331,True,True,CRC_P5


  
**2.**  
**nuclei_grouped_UMI_count.h5ad**  
Structure: nucleus_IDs x embedding diension (i.e. length == no. of all possible expressed genes in nucleus)

- `.X`: unique nucleus IDs (row ID) x genes (**column**: gene ID, **cell**: summed expression of that gene over all bins in the nucleus)
- `obs`: nucleus ID, Slide_ID
- `.var`: gene ID, feature type, genome

In [5]:
nuclei_grouped_UMI_count = sc.read_h5ad(os.path.join(data_path, "nuclei_grouped_UMI_count.h5ad"))
nuclei_grouped_UMI_count.obs

/home/charlottewolf/miniconda3/envs/NucleiSeg/lib/python3.11/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


,id,Slide_ID
0,ID_9187,CRC_P2
1,ID_9205,CRC_P2
2,ID_9225,CRC_P2
3,ID_9268,CRC_P2
4,ID_9275,CRC_P2
...,...,...
305615,ID_838150,CRC_P2
305616,ID_838165,CRC_P2
305617,ID_838170,CRC_P2
305618,ID_838173,CRC_P2


**3.**  
**count_area_filtered_adata.h5ad**  
Builds on the previous anndata, but applies filters both to *nucleus area* and *gene expression*.  
We remove nuclei with an area larger than a threshold (i.e. 500), to avoid broken or merged nuclei.  
We also remove nuclei with a UMI count below a threshold. --> avoid empty or low-quality nuclei  
- `.X`: same as `nuclei_grouped_UMI_count`
- `.obs`: 'id', 'Slide_ID', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'clusters'
- `.var`: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
- `uns`: 'clusters', 'hvg', 'log1p', 'neighbors', 'pca'
- `obsm`: 'X_pca'
- `varm`: 'PCs'
- `obsp`: 'connectivities', 'distances'

In [7]:
count_area_filtered_adata = sc.read_h5ad(os.path.join(data_path, "count_area_filtered_adata.h5ad"))
count_area_filtered_adata.obs

/home/charlottewolf/miniconda3/envs/NucleiSeg/lib/python3.11/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


,id,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,clusters,Slide_ID
0,ID_9187,335,5.817111,376.0,5.932245,24.202128,37.500000,64.095745,100.000000,5,CRC_P2
1,ID_9205,438,6.084499,507.0,6.230481,23.471400,33.333333,53.057199,100.000000,5,CRC_P2
2,ID_9225,867,6.766192,1139.0,7.038784,22.827041,31.606673,41.439860,67.778753,5,CRC_P2
3,ID_9268,182,5.209486,204.0,5.323010,35.294118,59.803922,100.000000,100.000000,0,CRC_P2
4,ID_9275,207,5.337538,240.0,5.484797,34.583333,55.416667,97.083333,100.000000,5,CRC_P2
...,...,...,...,...,...,...,...,...,...,...,...
305614,ID_838110,245,5.505332,281.0,5.641907,30.604982,48.398577,83.985765,100.000000,5,CRC_P2
305615,ID_838150,214,5.370638,245.0,5.505332,33.061224,53.469388,94.285714,100.000000,0,CRC_P2
305616,ID_838165,121,4.804021,126.0,4.844187,43.650794,83.333333,100.000000,100.000000,0,CRC_P2
305617,ID_838170,228,5.433722,255.0,5.545177,30.196078,49.803922,89.019608,100.000000,5,CRC_P2


In [ ]:
count_area_filtered_adata.var_names[count_area_filtered_adata.var_names.duplicated()]

Index(['TBCE', 'HSPA14', 'TMSB15B'], dtype='object')

In [ ]:
count_area_filtered_adata.var[count_area_filtered_adata.var_names == "TBCE"]

,gene_ids,feature_types,genome,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts,highly_variable,means,dispersions,dispersions_norm
TBCE,ENSG00000285053,Gene Expression,GRCh38,237,0.004021,0.004013,99.599608,238.0,5.476464,False,0.003434,0.002397,-0.343069
TBCE,ENSG00000284770,Gene Expression,GRCh38,191,0.003244,0.003238,99.677321,192.0,5.262690,False,0.002547,-0.053013,-0.694714


**4.**  
**patch_assigned_count_area_filtered_adata.h5ad**  
Completely same structure as count_area_filtered_adata.h5ad. 
However, .obs now has a column that assigns it to a certain image patch from the original image.

- `.X`:
- `.obs`: 'id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'clusters', 'Slide_ID', 'Patch_ID'
- `.var`: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
- `.obsm`: 'X_pca'
- `.varm`: 'PCs'

In [ ]:
patch_assigned_count_area_filtered_adata = sc.read_h5ad(os.path.join(data_path2, "patch_assigned_count_area_filtered_adata.h5ad"))
patch_assigned_count_area_filtered_adata

**5.**  
**nuclei_geometry.parquet**
Contains nuclei geometry for each nucleus.

Structure columns: nucleus ID, geometry, Slide_ID


In [8]:
nuclei_geometry = pd.read_parquet(os.path.join(data_path, "nuclei_geometry.parquet"))
nuclei_geometry

,id,geometry,Slide_ID
3,ID_1040398,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,CRC_P5
4,ID_1040399,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,CRC_P5
9,ID_1040404,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,CRC_P5
10,ID_1040405,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,CRC_P5
11,ID_1040409,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,CRC_P5
...,...,...,...
270971,ID_1773080,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...",CRC_P5
270972,ID_1773081,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,CRC_P5
270974,ID_1773083,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,CRC_P5
270976,ID_1773086,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...",CRC_P5


# SpaGCN results

In [2]:
import pandas as pd
import scanpy as sc

spagcn = sc.read_h5ad("Baselines/SpaGCN/results/CRC_P2/results_cluster_7.h5ad")
spagcn

AnnData object with n_obs × n_vars = 144296 × 17807
    obs: 'x1', 'x_array', 'y_array', 'x_pixel', 'y_pixel', 'pred', 'refined_pred'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'genename'
    uns: 'clusters', 'hvg', 'log1p', 'neighbors', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [4]:
spagcn.obs["refined_pred"].unique()

[0, 5, 1, 4, 3, 6, 2]
Categories (7, int64): [0, 1, 2, 3, 4, 5, 6]

In [5]:
spagcn.obs

,x1,x_array,y_array,x_pixel,y_pixel,pred,refined_pred
id,,,,,,,
ID_9187,1,519,2532,44611.827182,3936.127186,0,0
ID_9205,1,508,2532,44535.136268,3940.559428,5,5
ID_9225,1,511,2530,44551.987946,3951.589978,5,5
ID_9268,1,523,2533,44638.653759,3934.697314,0,0
ID_9275,1,514,2532,44576.256291,3938.473171,5,5
...,...,...,...,...,...,...,...
ID_838110,1,3290,99,64614.458946,21967.865712,5,5
ID_838150,1,3270,115,64466.349010,21850.096671,1,1
ID_838165,1,3296,53,64654.301892,22301.626635,0,5


# FIGURE OUT THESE SHITTY SVGs

In [3]:
import anndata
import gc
import geopandas as gpd
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
import scanpy as sc
import squidpy as sq

from csbdeep.utils import normalize
from matplotlib.colors import ListedColormap
from scipy import sparse
from shapely.geometry import Polygon, Point


filepath = Path("data/combined/01_preprocessing/cell_seg_stardist/002um/custom_binning")


In [4]:
file = "count_area_filtered_adata.h5ad"
adata_c = sc.read_h5ad(filepath / file)
adata_c

AnnData object with n_obs × n_vars = 456881 × 18085
    obs: 'id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'clusters', 'Slide_ID'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'moranI', 'moranI_pval_fdr', 'spatially_variable'
    uns: 'pca_loadings_per_sample', 'pca_variance_ratio_per_sample', 'sample_order'
    obsm: 'X_pca_per_sample_raw', 'spatial'
    varm: 'dispersions_norm_per_sample', 'dispersions_per_sample', 'highly_variable_per_sample', 'log1p_mean_counts_per_sample', 'log1p_total_counts_per_sample', 'mean_counts_per_sample', 'means_per_sample', 'moranI_per_sample', 'moranI_pval_per_sample', 'n_cells_by_counts_per_sample', 'pct_

In [6]:
adata_c.varm["highly_variable_per_sample"].shape

(18085, 5)

In [ ]:
file = "filtered_custom_bins.h5ad"
adata_f = sc.read_h5ad(filepath / file)

In [ ]:
file = "nuclei_grouped_UMI_count.h5ad"
adata_n = sc.read_h5ad(filepath / file)

In [7]:
file = "nuclei_geometry.parquet"
geo = gpd.read_parquet(filepath / file)
geo

,id,geometry,Slide_ID,centroid,interior_centroid
0,ID_100013,"POLYGON ((16412.747 16770.000, 16411.831 16771...",CRC_P1,POINT (16402.898 16771.658),POINT (16402.688 16771.822)
1,ID_100015,"POLYGON ((15099.245 14928.000, 15101.909 14931...",CRC_P1,POINT (15087.663 14927.799),POINT (15087.165 14927.117)
2,ID_100016,"POLYGON ((15808.291 14890.000, 15808.276 14891...",CRC_P1,POINT (15801.839 14891.113),POINT (15801.950 14891.725)
3,ID_100017,"POLYGON ((16733.960 15898.000, 16733.509 15898...",CRC_P1,POINT (16724.426 15893.065),POINT (16724.564 15893.277)
4,ID_100032,"POLYGON ((16487.416 16810.000, 16487.168 16811...",CRC_P1,POINT (16479.073 16813.421),POINT (16478.406 16813.951)
...,...,...,...,...,...
456876,ID_666491,"POLYGON ((52314.505 45776.000, 52314.601 45776...",Normal_P5,POINT (52310.871 45774.543),POINT (52311.142 45774.228)
456877,ID_666492,"POLYGON ((51664.354 46284.000, 51664.362 46284...",Normal_P5,POINT (51656.120 46288.063),POINT (51655.758 46288.416)
456878,ID_666496,"POLYGON ((52024.459 46084.000, 52024.903 46084...",Normal_P5,POINT (52011.773 46081.971),POINT (52011.576 46081.873)
456879,ID_666497,"POLYGON ((51702.852 45716.000, 51702.877 45716...",Normal_P5,POINT (51697.223 45714.807),POINT (51697.149 45714.896)


In [9]:
file = "patch_assigned_count_area_filtered_adata.h5ad"
adata_p = sc.read_h5ad(filepath / file)

/data/wv377102/miniconda3/envs/nuclei-seg/lib/python3.11/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [10]:
adata_p

AnnData object with n_obs × n_vars = 456881 × 18085
    obs: 'id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'clusters', 'Slide_ID', 'Patch_ID'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    obsm: 'X_pca'
    varm: 'PCs'

In [11]:
adata_c

AnnData object with n_obs × n_vars = 456881 × 18085
    obs: 'id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'clusters', 'Slide_ID'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'moranI', 'moranI_pval_fdr', 'spatially_variable'
    uns: 'pca_loadings_per_sample', 'pca_variance_ratio_per_sample', 'sample_order'
    obsm: 'X_pca_per_sample_raw', 'spatial'
    varm: 'dispersions_norm_per_sample', 'dispersions_per_sample', 'highly_variable_per_sample', 'log1p_mean_counts_per_sample', 'log1p_total_counts_per_sample', 'mean_counts_per_sample', 'means_per_sample', 'moranI_per_sample', 'moranI_pval_per_sample', 'n_cells_by_counts_per_sample', 'pct_